<a href="https://colab.research.google.com/github/turingvsclarke/MachineLearningProjects/blob/main/FinishedTextClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
# The tf-nightly installation caused dependency conflicts, leading to an ImportError.
# Relying on the stable TensorFlow version pre-installed in Colab.
import tensorflow as tf
import pandas as pd
# Use tf.keras for Keras components to avoid potential version conflicts with a standalone Keras installation.
from tensorflow import keras
from tensorflow.keras import layers
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.layers import TextVectorization
import numpy as np
import matplotlib.pyplot as plt
import os
import csv
print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path='train-data.tsv'
test_file_path = "valid-data.tsv"

In [ ]:
batch_size = 128
epochs = 10
seed=42

In [ ]:
### Data cleaning
train_dataset=pd.read_csv(train_file_path,sep='\t',header=None)
train_dataset=train_dataset.rename(columns={0:'label',1:'text'})
train_labels=train_dataset.copy().pop('label');
test_dataset=pd.read_csv(test_file_path,sep='\t',header=None)
test_dataset=test_dataset.rename(columns={0:'label',1:'text'})
test_labels=test_dataset.copy().pop('label')

In [ ]:
train_labels=train_labels.replace(['ham','spam'],[0,1])
test_labels=test_labels.replace(['ham','spam'],[0,1])

In [ ]:
test_labels

In [ ]:
embedding_layer=tf.keras.layers.Embedding(1000,5);

In [ ]:
### Defining the vocab size and number of words in a sequence
vocab_size=10000;
sequence_length=1000;

vectorize_layer = TextVectorization(
  max_tokens=vocab_size,
  output_mode='int',
  output_sequence_length=sequence_length
)
#train_dataset=train_dataset.copy().pop('text').astype(str).to_numpy()
#test_dataset=test_dataset.copy().pop('text').astype(str).to_numpy()
vectorize_layer.adapt(train_dataset['text'])
vectorize_layer.adapt(test_dataset['text'])

In [ ]:
#embedding=Embedding(input_dim=vocab_size,output_dim=16,name='embedding')
#print(embedding(vectorize_layer(train_dataset)))

In [ ]:
### Creating the model here. It will be a keras sequential model. We will fine tune to get the desired accuracy
model = keras.Sequential([
    vectorize_layer,
    Embedding(vocab_size,64,name='embedding'),
    keras.layers.Bidirectional(keras.layers.LSTM(64,  return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(32)),
    Dense(64,activation='relu'),
    keras.layers.Dropout(0.3),
    Dense(1,activation='sigmoid')
]);
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True)
)

In [ ]:
train_dataset['label'].value_counts()

In [ ]:
### Fit the model in keras
model.fit(train_dataset['text'].to_numpy(),train_labels,batch_size=batch_size,epochs=epochs,validation_data=(test_dataset['text'].to_numpy(),test_labels))


In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  list1=[]
  list1.append(pred_text)
  prediction=model.predict(np.array(list1).astype(np.object_))[0][0]
  if prediction<.5:
    label='ham';
  else:
    label='spam';
  return [prediction.item(),label]

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    print(prediction)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
